# 快速上手 npugraph_ex

环境验证 → 编写 Model → torch.compile 编译

### 使用说明

- **使用场景**：当前版本的TorchAir作为**beta特性**，主要专注于**推理场景**下的模型优化。
- **产品支持情况**：
    - Ascend 950PR/Ascend 950DT
    - Atlas A3 训练系列产品/Atlas A3 推理系列产品
    - Atlas A2 训练系列产品/Atlas A2 推理系列产品

- **整体约束**：PyTorch图模式支持单进程和多进程，每个进程**只支持使用1张NPU卡**，不支持使用多张NPU卡。
- **npugraph\_ex后端功能约束**：
    - 主要面向在线推理场景，暂不支持反向流程Capture成图、随机数算子Capture。
    - npugraph\_ex与torch.cuda.CUDAGraph原生接口（参见[Ascend Extension for PyTorch文档中心](https://hiascend.com/document/redirect/pytorchuserguide)中的《PyTorch 原生API支持度》“torch.cuda”）功能类似，约束与其保持一致（如不支持stream sync、动态控制流等），此处不再赘述。

### 安装

目前TorchAir暂未提供独立软件包，而是作为Ascend Extension for PyTorch的三方库，随着torch\_npu包一起发布。请直接安装torch\_npu插件，即可使用TorchAir。

## 1. 环境验证

首先确认 torch_npu 可用，CANN 版本满足要求。

In [ ]:
import torch
import torch_npu

print(f"PyTorch version: {torch.__version__}") # 打印Pytorch基础版本号
print(f"torch_npu version: {torch_npu.__version__}")
print(f"NPU available: {torch_npu.npu.is_available()}")
print(f"NPU device count: {torch_npu.npu.device_count()}")
print(f"NPU device name: {torch_npu.npu.get_device_name(0)}")

## 2. 第一个 npugraph_ex 程序

定义一个简单模型，使用 `backend="npugraph_ex"` 编译并执行。

In [ ]:
import torch
import torch_npu
from torch_npu.profiler import profile, ProfilerActivity
from torch.profiler import tensorboard_trace_handler

class SimpleModel(torch.nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x, y):
        h1 = torch.add(x, y)
        h2 = torch.add(h1, x)
        h3 = torch.add(h2, y)
        return h3

# 实例化模型，并将模型迁移到NPU设备上
model = SimpleModel().npu()

# 使用 npugraph_ex 后端进行整图编译
opt_model = torch.compile(model,backend="npugraph_ex",fullgraph=True,dynamic=False)

x = torch.randn(2, 64).npu()
y = torch.randn(2, 64).npu()

# ===================== Profiling性能采集（可选） =====================
prof_save_path = "./prof_npugraph"
with profile(
    activities = [ProfilerActivity.CPU, ProfilerActivity.NPU],
    record_shapes = False,
    with_stack = False,
    profile_memory = True,
    schedule = torch_npu.profiler.schedule(wait = 0, active = 2, warmup = 0, repeat = 1),
    on_trace_ready = tensorboard_trace_handler(prof_save_path)
) as prof:

    # 首次执行会触发 Capture + Compile
    out = opt_model(x, y)

print(f"Output shape: {out.shape}")
print(f"Output values:\n{out}")
print(f"\nProfiling数据已导出至目录：{prof_save_path}")

torch.compile为PyTorch原生接口，接口详细介绍请参见[官网](https://pytorch.org/docs/stable/generated/torch.compile.html#torch-compile)，接口原型如下：

```python
torch.compile(model=None, *, fullgraph=False, dynamic=None, backend='npugraph_ex', mode=None, options=None, disable=False)
```

torch.compile参数配置说明参见[表1](#fig1)。





**表 1**  torch.compile参数说明（aclgraph模式）<a id="fig1"></a>

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr>
    <th align="left">参数名</th>
    <th align="left">PyTorch原生参数说明</th>
    <th align="left">aclgraph模式下参数说明</th>
  </tr>
  <tr>
    <td align="left">model</td>
    <td align="left"><strong>必选参数</strong>。入图部分的模型或者函数。</td>
    <td align="left">与原生含义一致。</td>
  </tr>
  <tr>
    <td align="left">fullgraph</td>
    <td align="left">可选参数，bool类型。是否捕获整图进行优化。<br>False（缺省值）：非整图优化。<br>True：捕获整图优化。</td>
    <td align="left">建议设置为True，要求将整个函数或模型捕获到一个单一的计算图中。如果编译器遇到无法追踪到该单一图中的代码时（即“图中断”），则会引发错误。</td>
  </tr>
  <tr>
    <td align="left">dynamic</td>
    <td align="left">可选参数，bool类型或None。是否启用动态Shape追踪。<br>None（缺省值）：自动检测是否启用动态Shape追踪。<br>False：不启用动态Shape追踪。<br>True：启用动态Shape追踪。</td>
    <td align="left">与原生含义一致。</td>
  </tr>
  <tr>
    <td align="left">backend</td>
    <td align="left"><strong>必选参数</strong>，后端选择，缺省值为"inductor"。</td>
    <td align="left">需显式传入backend="npugraph_ex"。</td>
  </tr>
  <tr>
    <td align="left">mode</td>
    <td align="left">开销模式，内存开销模式选择，缺省值为None。</td>
    <td align="left">昇腾NPU<strong>暂不支持</strong>。</td>
  </tr>
  <tr>
    <td align="left">options</td>
    <td align="left">优化选项，缺省值为None。</td>
    <td align="left">提供多种基础功能配置。</td>
  </tr>
  <tr>
    <td align="left">disable</td>
    <td align="left">可选参数，bool类型。是否关闭torch.compile能力。<br>False（缺省值）：开启torch.compile能力。<br>True：关闭torch.compile能力，采用单算子模式。</td>
    <td align="left">与原生含义一致。</td>
  </tr>
</table>


## 3.课后习题
（1）【单选题】基于 npugraph_ex 快速上手相关知识，下列说法错误的是？
- A. TorchAir 无独立安装包，安装 torch_npu 后即可直接使用 npugraph_ex 后端
- B. 使用 npugraph_ex 时，torch.compile必须显式指定backend="npugraph_ex"
- C. npugraph_ex 支持模型反向传播流程捕获入图，可用于训练场景图加速
- D. 当前 TorchAir beta 版本，单进程仅支持单 NPU 卡，不支持多卡并行图捕获

（2）【单选题】使用 torch.compile 开启 npugraph_ex 图模式，必须指定的 backend 参数是？
- A. backend="inductor"
- B. backend="npu"
- C. backend="npugraph_ex"
- D. backend="ascend"

（3）【单选题】开启 npugraph_ex 整图捕获、禁止图中断，需要配置哪个参数为 True？
- A. dynamic=True
- B. fullgraph=True
- C. disable=True
- D. mode="max-autotune"

（4）【单选题】关于 npugraph_ex 首次推理与二次推理耗时差异，下列说法正确的是？（）
- A. 两次推理耗时基本一致
- B. 首次推理耗时更低，无初始化开销
- C. 首次包含 Capture 捕获开销，耗时高；后续 Replay 回放耗时极低
- D. 二次推理会重新捕获计算图

（5）【实验题】请基于本章 SimpleModel 示例代码完成Eager单算子执行模式

**运行以下代码单元查看单选题答案与解析，[实验题答案解析](./answer/02.02_answer.ipynb)**

In [ ]:
import os
answer_path = "answer/02.02_answer.txt"
if os.path.exists(answer_path):
    with open(answer_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("答案文件未找到，请检查 ./answer/ 目录结构。")